# 01 — Data verification

This is the "genuine data verification". Three parts:

1. **Fingerprint every raw file** (hash + timestamp) 
2. **Schema / sanity checks** — catches corruption, not just missing values.
3. **Cross-table consistency checks** — do the IDs referenced in `funding_rounds`/`acquisitions`/`ipos` actually exist in `objects_slim.csv`?
4. **Random spot-check sample** — pulls real companies for you to manually verify against Crunchbase.com or press coverage. This produces the "% matched real records" number for your report.

Output: `reports/data_manifest.json` and `reports/verification_sample.csv`.

In [1]:
import pandas as pd
import hashlib
import json
import time
import os

RAW = "../data/raw"
REPORTS = "../reports"
os.makedirs(REPORTS, exist_ok=True)

FILES = ["objects_slim.csv", "funding_rounds.csv", "acquisitions.csv", "ipos.csv"]

## 1. Fingerprint each file (SHA-256 hash + timestamp)

In [2]:
def sha256_of_file(path, block_size=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(block_size), b""):
            h.update(block)
    return h.hexdigest()

manifest = {}
for fname in FILES:
    path = f"{RAW}/{fname}"
    manifest[fname] = {
        "sha256": sha256_of_file(path),
        "size_bytes": os.path.getsize(path),
        "verified_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    print(f"[hash] {fname}: {manifest[fname]['sha256'][:16]}...  ({manifest[fname]['size_bytes']:,} bytes)")

with open(f"{REPORTS}/data_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)
print(f"\n[done] wrote {REPORTS}/data_manifest.json — cite this hash in your report as proof the file used is fixed and traceable")

[hash] objects_slim.csv: 24c4cdc79f43020e...  (13,747,419 bytes)
[hash] funding_rounds.csv: f622ea495d1909cf...  (13,134,510 bytes)
[hash] acquisitions.csv: 5a89ca0f75502c33...  (2,099,350 bytes)
[hash] ipos.csv: 0e610556b450b9bf...  (143,038 bytes)

[done] wrote ../reports/data_manifest.json — cite this hash in your report as proof the file used is fixed and traceable


## 2. Schema / sanity checks

In [3]:
obj = pd.read_csv(f"{RAW}/objects_slim.csv", encoding="ISO-8859-1", low_memory=False)
fr = pd.read_csv(f"{RAW}/funding_rounds.csv", encoding="ISO-8859-1", low_memory=False)
acq = pd.read_csv(f"{RAW}/acquisitions.csv", encoding="ISO-8859-1", low_memory=False)
ipo = pd.read_csv(f"{RAW}/ipos.csv", encoding="ISO-8859-1", low_memory=False)

checks = []

# dates parse cleanly?
for df_name, df, col in [("objects", obj, "founded_at"), ("objects", obj, "closed_at"),
                          ("funding_rounds", fr, "funded_at"), ("acquisitions", acq, "acquired_at"),
                          ("ipos", ipo, "public_at")]:
    parsed = pd.to_datetime(df[col], errors="coerce")
    bad = parsed.isna().sum() - df[col].isna().sum()  # newly-unparseable, i.e. malformed not just missing
    checks.append((f"{df_name}.{col}", "malformed dates", bad))

# negative or impossible values
neg_funding = (fr["raised_amount_usd"].fillna(0) < 0).sum()
checks.append(("funding_rounds.raised_amount_usd", "negative values", neg_funding))

future_founded = (pd.to_datetime(obj["founded_at"], errors="coerce") > pd.Timestamp.now()).sum()
checks.append(("objects.founded_at", "founding dates in the future", future_founded))

closed_before_founded = (
    (pd.to_datetime(obj["closed_at"], errors="coerce") < pd.to_datetime(obj["founded_at"], errors="coerce")).sum()
)
checks.append(("objects", "closed_at earlier than founded_at", closed_before_founded))

print(f"{'check':45s} {'issue':30s} {'count'}")
for name, issue, count in checks:
    flag = "  <-- investigate" if count > 0 else ""
    print(f"{name:45s} {issue:30s} {count}{flag}")

check                                         issue                          count
objects.founded_at                            malformed dates                0
objects.closed_at                             malformed dates                0
funding_rounds.funded_at                      malformed dates                0
acquisitions.acquired_at                      malformed dates                0
ipos.public_at                                malformed dates                0
funding_rounds.raised_amount_usd              negative values                0
objects.founded_at                            founding dates in the future   0
objects                                       closed_at earlier than founded_at 44  <-- investigate


## 3. Cross-table consistency (foreign-key style checks)

In [4]:
known_ids = set(obj["id"])

fr_orphans = (~fr["object_id"].isin(known_ids)).sum()
acq_orphans = (~acq["acquired_object_id"].isin(known_ids)).sum()
ipo_orphans = (~ipo["object_id"].isin(known_ids)).sum()

print(f"[consistency] funding_rounds rows pointing to unknown companies: {fr_orphans:,} / {len(fr):,}")
print(f"[consistency] acquisitions rows pointing to unknown companies:   {acq_orphans:,} / {len(acq):,}")
print(f"[consistency] ipos rows pointing to unknown companies:           {ipo_orphans:,} / {len(ipo):,}")
print("\nA small orphan count is normal (rounding/versioning across the original Crunchbase export).")
print("A LARGE orphan count means something is broken in how you loaded/matched the files — stop and check before modeling.")

[consistency] funding_rounds rows pointing to unknown companies: 302 / 52,928
[consistency] acquisitions rows pointing to unknown companies:   18 / 9,562
[consistency] ipos rows pointing to unknown companies:           20 / 1,259

A small orphan count is normal (rounding/versioning across the original Crunchbase export).
A LARGE orphan count means something is broken in how you loaded/matched the files — stop and check before modeling.


## 4. Random spot-check sample

This is the part you do by hand: for each company below, search its name on
Crunchbase.com, Google, or press coverage and check whether the status,
sector and rough funding total match. Fill in the `matches_public_record`
column yourself, then compute the match rate — that's your citable
verification number.

In [5]:
sample = obj[obj["status"].notna() & obj["category_code"].notna() & obj["funding_total_usd"].notna()] \
    .sample(n=20, random_state=42)[["id", "name", "category_code", "status",
                                     "founded_at", "closed_at", "funding_total_usd"]] \
    .reset_index(drop=True)

sample["matches_public_record"] = ""  # fill in: yes / no / can't verify, after checking manually
sample.to_csv(f"{REPORTS}/verification_sample.csv", index=False)
print(f"[done] wrote {REPORTS}/verification_sample.csv — {len(sample)} companies to manually check")
sample

[done] wrote ../reports/verification_sample.csv — 20 companies to manually check


,id,name,category_code,status,founded_at,closed_at,funding_total_usd,matches_public_record
0,c:189169,OK YO,ecommerce,operating,2009-08-01,NaN,0.0,
1,c:266748,Nice Entertainment Group,games_video,acquired,1969-01-01,NaN,0.0,
2,c:44771,Boxcar,software,acquired,2009-06-23,NaN,150000.0,
3,c:7733,blogrunner,web,acquired,NaN,NaN,0.0,
4,c:253253,Spoonjuice,mobile,operating,NaN,NaN,0.0,
5,c:76820,Docudesk,software,operating,2001-01-01,NaN,0.0,
6,c:171048,Attender,mobile,operating,2011-08-01,NaN,25000.0,
7,c:28494,Vebnet Holdings,software,acquired,2000-01-01,NaN,0.0,
8,c:31701,Tube2Tone,mobile,operating,2009-08-19,NaN,90000.0,
9,c:282424,Eta Consults,consulting,operating,2011-11-01,NaN,0.0,


In [8]:
# After you've filled in matches_public_record by hand, rerun this cell:
filled = pd.read_csv(f"{REPORTS}/verification_sample.csv")
checked = filled[filled["matches_public_record"].isin(["yes", "no"])]
if len(checked) > 0:
    match_rate = (checked["matches_public_record"] == "yes").mean() * 100
    print(f"[verification] {match_rate:.0f}% of {len(checked)} manually-checked companies matched public record")
else:
    print("[verification] fill in the matches_public_record column first, then rerun this cell")

[verification] 100% of 15 manually-checked companies matched public record
